# Xselo no Kaggle (de graça)

O Kaggle (do Google) dá **~30 horas de GPU por semana, de graça**: duas placas T4 que, juntas, seguram o Xselo 31B. É mais devagar que a A100 do Colab (uma resposta média leva de meio minuto a um minuto e pouco), mas não gasta unidade nenhuma. Usando 2 a 3 horas por dia, sobra quota.

## Preparação (uma vez só)

1. **Conta:** crie uma em kaggle.com e confirme o telefone em *Settings → Phone Verification*. Sem isso, o Kaggle não libera GPU nem internet.
2. **Suba o Xselo como dataset privado:** pegue o `.zip` que o Colab baixou no fim (célula 8), ou a pasta do modelo no seu Drive (`MyDrive/Ratex/xselo-0-5-gemma31b-...`). No Kaggle: *Create → New Dataset*, arraste o `.zip`, dê o nome `xselo`, deixe **Private** e clique em *Create*. O Kaggle descompacta sozinho.
3. **Abra este notebook no Kaggle:** *Create → New Notebook*, depois *File → Import Notebook* e cole o link
   `https://github.com/SeniorVortex/Ratex/blob/main/notebooks/xselo_no_kaggle.ipynb`
   (se não aceitar o link, baixe o arquivo pelo GitHub e faça o upload dele).
4. **No painel da direita (Session options):**
   - *Accelerator* → **GPU T4 x2**
   - *Internet* → **On**
   - *Add Input* → aba **Datasets** → *Your Work* → o seu `xselo`
   - *Add Input* → aba **Models** → procure **Gemma 4** → variação **Transformers** → **gemma-4-31b-it**. Assim a Gemma já vem pronta, sem baixar 60 GB.

## Toda vez

Escolha o `MODO` na célula 1 e clique em **Run All** (ou *Run → Run all*). Quando terminar, **pare a sessão** (botão de power no alto, *Stop Session*): a quota só gasta com a sessão ligada.

**`MODO = "api"`** (o padrão): o Xselo vira uma API igual à da OpenAI, com um link público. A última célula mostra três coisas pra colar em qualquer site ou app que aceite *OpenAI compatible* / *custom endpoint* / *reverse proxy* (SillyTavern, Open WebUI, Janitor AI, extensões...):

- **URL base:** `https://algum-nome.trycloudflare.com/v1` (muda toda vez que liga)
- **API key:** a chave mostrada (ou a sua, se guardar em *Add-ons → Secrets* como `XSELO_CHAVE`)
- **Modelo:** `xselo-0.5`

Ligue o **streaming** no site, se tiver a opção: o texto vai aparecendo enquanto ele escreve. Se o site pedir o tamanho do contexto (*Context Size*), coloque **4000**; e o tamanho da resposta, uns **400 a 600** tokens. O link só funciona enquanto essa célula estiver rodando.

**`MODO = "chat"`**: conversa aqui mesmo, na caixinha `você>`. **`MODO = "discord"`**: liga o bot do Discord.

- `/foto <link da imagem> <pergunta>` manda uma imagem (um link direto de imagem da internet, ou o caminho de um arquivo que você subiu como dataset).
- `/lembrancas`, `/lembrar <fato>`, `/esquecer`, `/novo` funcionam igual ao Colab. As lembranças ficam só enquanto a sessão estiver aberta.
- No modo `discord`, o token vai em *Add-ons → Secrets* com o nome `DISCORD_TOKEN`.

In [ ]:
# ⚙️ 1. Configuração
PRECISAO = "float32"        # "float32" = seguro pra Gemma nas T4 | "float16" = um pouco mais leve, mas a Gemma pode dar erro
MODO = "api"                # "api" = link pra sites (OpenAI compatível) | "chat" = conversar aqui | "discord" = bot
SYSTEM_DO_SITE = "juntar"   # modo api: o system prompt do site entra "juntar" com o do Xselo, ou "substituir" (bom pra personagens)
CONTEXTO = 4000             # modo api: máximo de tokens de conversa que entra (o mais antigo é cortado). Deu "out of memory"? diminua
BUSCA_NA_INTERNET = True    # pesquisa no SearXNG quando a pergunta é sobre coisa atual (ou começa com /buscar)

In [ ]:
# 📦 2. Código do Ratex e bibliotecas (antes de qualquer coisa carregar o torch)
import os, subprocess, sys
if os.path.isdir("/kaggle/working/Ratex/.git"):
    !git -C /kaggle/working/Ratex pull -q
else:
    !git clone -q --depth 1 https://github.com/SeniorVortex/Ratex.git /kaggle/working/Ratex
# tenta manter o torch que o Kaggle já traz (combina com as placas dele); se não der, deixa o pip escolher
from importlib.metadata import version as _versao
open("/tmp/manter_torch.txt", "w").write(f"torch=={_versao('torch')}\n")
pacotes = ["transformers>=5", "peft>=0.17", "accelerate", "bitsandbytes", "discord.py>=2.4", "fastapi", "uvicorn"]
feito = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "-c", "/tmp/manter_torch.txt", *pacotes])
if feito.returncode != 0:
    print("(não deu pra manter o torch do Kaggle; instalando a versão que as bibliotecas pedem)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pacotes], check=True)
!git -C /kaggle/working/Ratex log --oneline -1

teste = subprocess.run([sys.executable, "-c",
    "import sys, torch, transformers, peft, bitsandbytes; "
    "from transformers import AutoModelForImageTextToText, AutoProcessor; "
    "print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPUs', torch.cuda.device_count(), "
    "'| transformers', transformers.__version__, '| peft', peft.__version__)"],
    capture_output=True, text=True)
if teste.returncode != 0:
    print(teste.stderr[-2500:])
    raise SystemExit("❌ As bibliotecas não carregaram. Pare a sessão (Stop Session), ligue de novo e dê Run All. "
                     "Se continuar, mande o texto acima pro Claude.")
print("✅", teste.stdout.strip())

if "torch" in sys.modules:  # célula rodada de novo e a instalação trocou o torch: precisa reiniciar o Python
    from importlib.metadata import version
    if version("torch").split("+")[0] != sys.modules["torch"].__version__.split("+")[0]:
        print("🔄 Reiniciando o Python (o torch foi atualizado). Quando voltar, dê Run All de novo.")
        os.kill(os.getpid(), 9)

In [ ]:
# 🔎 3. Achar o Xselo, a Gemma e as GPUs
import os, sys, json
os.environ["PYTORCH_ALLOC_CONF"] = os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.chdir("/kaggle/working/Ratex"); sys.path.insert(0, "/kaggle/working/Ratex")
import torch
from nucleo.nuvem import achar_base, achar_xselo

n_gpus = torch.cuda.device_count()
if n_gpus == 0:
    raise SystemExit("Sem GPU! No painel da direita: Session options → Accelerator → GPU T4 x2.")
for i in range(n_gpus):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 2**30:.0f} GB)")
if n_gpus < 2:
    print("⚠️ Só 1 GPU: o Xselo 31B não cabe numa T4 sozinha. Escolha GPU T4 x2 no painel da direita.")

PASTA_XSELO = achar_xselo("/kaggle/input")
if PASTA_XSELO is None:
    raise SystemExit("Não achei o Xselo. Suba o .zip do modelo como dataset e adicione em Add Input → Datasets "
                     "(passo 2 e 4 da preparação, lá em cima).")
cfg_xselo = json.loads((PASTA_XSELO / "ratex_config.json").read_text(encoding="utf-8"))
BASE = achar_base("/kaggle/input", cfg_xselo["base"])
print("Xselo:", PASTA_XSELO)
if BASE is None:
    BASE = cfg_xselo["base"]
    print(f"⚠️ Não achei a {BASE} anexada: vou baixar do Hugging Face (~60 GB, demora e pode faltar disco).")
    print("   Melhor: Add Input → Models → Gemma 4 → Transformers → gemma-4-31b-it, e rode de novo.")
else:
    print("Base:", BASE)

In [ ]:
# 🧠 4. Carregar o Xselo (dividido nas duas GPUs; leva uns 5-10 min)
import os, time
os.environ["XSELO_DTYPE"] = PRECISAO
# nas T4, a parte que enxerga imagem (~2 GB) vai pra CPU: sobra GPU pra conversa (fotos ficam um pouco mais lentas)
os.environ["XSELO_VISAO_NA_CPU"] = "1"
from nucleo.hibrido import carregar_hibrido, enxerga
from nucleo.memoria import Memoria
t0 = time.time()
if "xselo" not in globals():
    xselo, tok, cfg = carregar_hibrido(PASTA_XSELO, device="cuda", base=str(BASE))
    memoria = Memoria()
print(f"✅ {cfg['nome']} carregado em {(time.time() - t0) / 60:.0f} min | enxerga imagens: {'sim' if enxerga(xselo) else 'não'}")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GB ocupados")

In [ ]:
# 🌐 5. Busca na internet (SearXNG; na 1ª vez leva 1-2 min)
busca = None
if BUSCA_NA_INTERNET:
    try:
        from nucleo.busca import Busca, iniciar_searxng
        busca = Busca(iniciar_searxng())
        print("✅ busca funcionando. O Xselo pesquisa sozinho quando a pergunta é sobre coisa atual; pra forçar, comece com /buscar")
    except Exception as erro:
        busca = None
        print(f"⚠️ não deu pra ligar a busca ({erro}). Tudo bem: o Xselo responde sem ela.")

In [ ]:
# 💬 6. MODO = "chat": conversar com o Xselo (/sair pra parar; /foto <link ou caminho> <pergunta>; /lembrancas; /esquecer)
from nucleo.hibrido import responder
from nucleo.lembrancas import Conversa, Lembrancas, resumidor
conversa = Conversa("voce", Lembrancas("/kaggle/working/lembrancas.json"), resumir=resumidor(xselo, tok))
if MODO != "chat":
    print(f'(modo "{MODO}": pulando o chat do notebook)')
while MODO == "chat":
    msg = input("você> ").strip()
    if msg in ("/sair", "sair", ""):
        break
    aviso = conversa.comando(msg)
    if aviso:
        print(aviso)
        continue
    imagens = []
    if msg.startswith("/foto"):
        partes = msg[len("/foto"):].strip().split(" ", 1)
        if not partes[0]:
            print("(use assim: /foto https://site.com/imagem.jpg o que é isso?)")
            continue
        imagens = [partes[0]]
        msg = partes[1].strip() if len(partes) > 1 else "O que você vê nessa imagem?"
    conversa.usuario(msg, imagens)
    print("xselo> ", end="")
    try:
        resposta = responder(xselo, tok, conversa.historico, system_prompt=conversa.system(cfg["system_prompt"]),
                             memoria=memoria, busca=busca, **cfg["geracao"])
    except Exception as erro:  # link de imagem quebrado, imagem num modelo que não enxerga...
        print(f"({erro})")
        conversa.historico.pop()
        continue
    conversa.xselo(resposta)
    print()

In [ ]:
# 🤖 7. MODO = "discord": bot do Discord (fica rodando até você parar a célula; o passo a passo está no bot_discord.py)
if MODO != "discord":
    print(f'(modo "{MODO}": bot do Discord desligado)')
else:
    from kaggle_secrets import UserSecretsClient
    import bot_discord
    from nucleo.lembrancas import Lembrancas
    token = UserSecretsClient().get_secret("DISCORD_TOKEN")
    bot = bot_discord.XseloDiscord(xselo, tok, cfg, memoria=memoria, busca=busca,
                                   lembrancas=Lembrancas("/kaggle/working/lembrancas_discord.json"))
    cliente = bot_discord.criar_cliente(bot)
    await cliente.start(token)

In [ ]:
# 🔌 8. MODO = "api": o Xselo com API igual à da OpenAI e link público (fica rodando até você parar a célula)
import secrets, time
if MODO != "api":
    print(f'(modo "{MODO}": API desligada)')
else:
    from servidor_api import abrir_tunel, criar_app, ligar_servidor
    try:
        from kaggle_secrets import UserSecretsClient
        CHAVE = UserSecretsClient().get_secret("XSELO_CHAVE")
    except Exception:
        CHAVE = "xselo-" + secrets.token_urlsafe(18)
    NOME_API = f"xselo-{cfg.get('versao', '')}"
    app = criar_app(xselo, tok, cfg, CHAVE, NOME_API, memoria=memoria, busca=busca, modo_system=SYSTEM_DO_SITE, contexto=CONTEXTO)
    ligar_servidor(app, 8000)
    URL = abrir_tunel(8000)
    print("✅ Xselo no ar! Cole isso no site/app (campo OpenAI compatible / custom endpoint / reverse proxy):\n")
    print(f"   URL base:  {URL}/v1")
    print(f"   API key:   {CHAVE}")
    print(f"   Modelo:    {NOME_API}\n")
    print("Não mostre a API key pra ninguém que você não queira usando a sua GPU. Pra desligar: ⏹ nesta célula.")
    inicio = time.time()
    try:
        while True:  # a célula continua rodando: é isso que mantém o Kaggle ligado e o link funcionando
            time.sleep(600)
            print(f"   ...no ar há {(time.time() - inicio) / 60:.0f} min")
    except KeyboardInterrupt:
        print("API desligada.")